### Dealing with real Data: DT's Applied to the Full HWC
#### Lecture/Lab 23, Phys 4730 F25

<br>

Ref: V. Acquaviva, <i>Machine Learning for Physics and Astronomy</i> 

In [ ]:
%reset -f

In [ ]:
import pandas as pd
import numpy as np

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

font = {'size'   : 14}
matplotlib.rc('font', **font)
matplotlib.rc('xtick', labelsize=14) 
matplotlib.rc('ytick', labelsize=14) 
matplotlib.rcParams['figure.dpi'] = 100

import sklearn.tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics 
from sklearn.model_selection import cross_val_predict, cross_val_score, cross_validate
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn import tree
from sklearn.tree import export_graphviz
from scipy import stats

import pandas as pd
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 100)

### Explore the full HWC data set


In [ ]:
### read the dataset, look at the first few rows ###

df = pd.read_csv('20240321_hwc_full.csv', sep = ',')
df.head()

In [ ]:
### Many more columns than our preliminary set! Print the full list... ###

df.columns

### Pare down the dataset to something more manageable

In [ ]:
### Create a new data frame called bindf, dropping the ###
### old habitability label which had 3 possibilities.  ###

bindf = df.drop('P_HABITABLE', axis = 1) 

In [ ]:
### Creating instead a new habitability label which is an OR  ###
### of "optimistically habitable" and "expected habitable".   ###
### Recast the label as an integer.                           ###

bindf['P_HABITABLE'] = (np.logical_or((df.P_HABITABLE == 1) , (df.P_HABITABLE == 2)))
bindf['P_HABITABLE'] = bindf['P_HABITABLE'].astype(int)
bindf.head()

In [ ]:
### Confine ourself to the 3(2) features and 1 target used previously ###

NFEATURES = 3

if(NFEATURES == 3):
    final_features = bindf[['S_MASS', 'P_PERIOD', 'P_DISTANCE']]
else:
    final_features = bindf[['S_MASS', 'P_PERIOD']]

final_features.head()

In [ ]:
targets = bindf.P_HABITABLE
targets.head()

### Now deal with some anomalies. 

In [ ]:
final_features.shape

In [ ]:
### get rid of any instance with at least one NaN in any column ###

final_features = final_features.dropna(axis = 0) #gets rid of any instance with at least one NaN in any column
final_features.shape

In [ ]:
# check for outliers in the features 

plt.figure(figsize=(8,8))

if(NFEATURES == 3):

    plt.subplot(221)
    plt.hist(final_features.iloc[:,0], bins = 100, alpha = 0.5,label="mass")
    plt.yscale("log")
    plt.legend()

    plt.subplot(222)
    plt.hist(final_features.iloc[:,1], bins = 100, alpha = 0.5,label="period")
    plt.yscale("log")
    plt.legend()

    plt.subplot(223)
    plt.hist(final_features.iloc[:,2], bins = 100, alpha = 0.5,label="distance")
    plt.yscale("log")
    plt.legend()

else:
    
    plt.subplot(211)
    plt.hist(final_features.iloc[:,0], bins = 100, alpha = 0.5,label="mass")
    plt.yscale("log")
    plt.legend()

    plt.subplot(212)
    plt.hist(final_features.iloc[:,1], bins = 100, alpha = 0.5,label="period")
    plt.yscale("log")
    plt.legend()

plt.show()

In [ ]:
### This eliminates objects with > 5 sigma outliers in any column;   ###
### it counts the number of standard deviations away from the mean,  ###
### so it might not be ideal.                                        ###

final_features = final_features[(np.abs(stats.zscore(final_features)) < 5).all(axis=1)]
final_features.shape

In [ ]:
targets = targets[final_features.index]
targets.shape

In [ ]:
### reset the indices of the features and targets

final_features = final_features.reset_index(drop=True)
targets = targets.reset_index(drop=True)

pd.concat([final_features, targets], axis=1)

### Now we can plot the data

In [ ]:
### make a feature-space scatter plot of our data, in the familiar form ###

plt.figure(figsize=(10,6))

cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", ['#20B2AA','#FF00FF'])

a = plt.scatter(final_features['S_MASS'], final_features['P_PERIOD'], \
                marker = '+', c=targets, s = 100, cmap=cmap)

plt.yscale('log')
plt.xlabel('Mass of Parent Star (Solar Mass Units)')
plt.ylabel('Period of Orbit (days)');

bluepatch    = mpatches.Patch(color='#20B2AA', label='Not Habitable')
magentapatch = mpatches.Patch(color='#FF00FF', label='Habitable')

plt.legend(handles=[magentapatch, bluepatch], loc = 'lower right', fontsize = 14);

plt.show()

In [ ]:
### Whoa... relatively few habitable planets!  ###
### The data set is unbalanced. Quantify this. ###

np.sum(targets)/len(targets)

### Now, do the decision tree training

In [ ]:
### a random train/test split will assign 25% of instances to test set

Xtrain, Xtest, ytrain, ytest = train_test_split(final_features, targets, random_state=2)
Xtrain.shape, Xtest.shape

In [ ]:
### choose Decision Tree Method, build the model ###

model = DecisionTreeClassifier(random_state=5)
model.fit(Xtrain, ytrain)

In [ ]:
### obtain information from the tree structure ###

tree0 = model.tree_
split_values = tree0.threshold
split_values

In [ ]:
### sklearn enables us to visualize the decision tree ###

plt.figure(figsize=(40,20))  # customize according to the size of your tree

tree.plot_tree(model, feature_names = ['Mass', 'Period', 'Distance'], class_names = ['Not Habitable','Habitable'])

plt.savefig("fulltree.png",dpi=300,facecolor='white', transparent=False)
plt.show()

In [ ]:
### check the accuracy scores ###

print("training set accuracy score:    %7.5f" % metrics.accuracy_score(ytrain, model.predict(Xtrain)))
print("testing set accuracy score:     %7.5f" % metrics.accuracy_score(ytest, model.predict(Xtest)))

In [ ]:
### sounds good, but suppose we had just classified everything as uninhabitable? ###

print("lazy classifier accuracy score: %7.5f" % metrics.accuracy_score(ytest, np.zeros(len(ytest))))